In [1]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

MODEL_DIR = Path("../models")

model_without_history = joblib.load(
    MODEL_DIR / "xgb_without_history_production.joblib"
)

model_with_history = joblib.load(
    MODEL_DIR / "xgb_with_history_production.joblib"
)

with open(
    MODEL_DIR / "production_metadata.json",
    "r"
) as f:
    model_metadata = json.load(f)

In [2]:
features_without_history = (
    model_metadata["features_without_history"]
)

features_with_history = (
    model_metadata["features_with_history"]
)

In [5]:
def predict_player_value(player_data):
    """
    Predict a player's market value using the appropriate production model.

    Parameters
    ----------
    player_data : dict
        Dictionary containing the player's engineered model features.
        If 'previous_market_value_in_eur' is available, the with-history
        model is used. Otherwise, the without-history model is used.

    Returns
    -------
    dict
        Model used and predicted market value.
    """

    player_data = player_data.copy()

    # check whether historical market value is available
    previous_value = player_data.get(
        "previous_market_value_in_eur"
    )

    has_history = (
        previous_value is not None
        and pd.notna(previous_value)
        and previous_value > 0
    )

    if has_history:

        model = model_with_history
        required_features = features_with_history
        model_used = "with_history"

    else:

        model = model_without_history
        required_features = features_without_history
        model_used = "without_history"

    # ensure all required model features are present
    missing_features = [
        feature
        for feature in required_features
        if feature not in player_data
    ]

    if missing_features:
        raise ValueError(
            f"Missing required features: {missing_features}"
        )

    # create one-row DataFrame in the exact feature order
    X_player = pd.DataFrame(
        [player_data]
    )[required_features]

    # model prediction is in log space
    predicted_log_value = model.predict(
        X_player
    )[0]

    # convert back to euros
    predicted_value_eur = np.expm1(
        predicted_log_value
    )

    return {
        "model_used": model_used,
        "predicted_market_value_eur": float(
            predicted_value_eur
        ),
        "predicted_market_value_m": float(
            predicted_value_eur / 1_000_000
        )
    }

In [ ]:
RATE_MINUTES_THRESHOLD = 900


def build_player_features(raw_data):
    """
    Convert raw current-season player data into the engineered
    features expected by the production models.

    Parameters
    ----------
    raw_data : dict
        Raw/aggregated player-season information.

    Returns
    -------
    dict
        Model-ready player features.
    """

    features = raw_data.copy()

    minutes = features.get("total_minutes_played", np.nan)
    games = features.get("total_games_played_season", np.nan)

    # --------------------------------------------------
    # Per-90 features
    # --------------------------------------------------

    def calculate_per_90(total):
        if (
            pd.notna(minutes)
            and minutes >= RATE_MINUTES_THRESHOLD
            and pd.notna(total)
        ):
            return total * 90 / minutes

        return np.nan

    features["goals_per_90_min"] = calculate_per_90(
        features.get("total_goals")
    )

    features["assists_per_90_min"] = calculate_per_90(
        features.get("total_assists")
    )

    total_goal_contributions = (
        features.get("total_goals", 0)
        + features.get("total_assists", 0)
    )

    features["goal_contributions_per_90_min"] = (
        calculate_per_90(total_goal_contributions)
    )

    features["yellow_cards_per_90_min"] = calculate_per_90(
        features.get("total_yellow_cards")
    )

    features["red_cards_per_90_min"] = calculate_per_90(
        features.get("total_red_cards")
    )

    # --------------------------------------------------
    # Clean-sheet rate
    # --------------------------------------------------

    clean_sheets = features.get(
        "total_clean_sheets",
        np.nan
    )

    if (
        pd.notna(minutes)
        and minutes >= RATE_MINUTES_THRESHOLD
        and pd.notna(games)
        and games > 0
        and pd.notna(clean_sheets)
    ):
        features["clean_sheet_rate"] = (
            clean_sheets / games
        )
    else:
        features["clean_sheet_rate"] = np.nan

    # --------------------------------------------------
    # Club points per game
    # --------------------------------------------------

    # Calculate only if it has not already been supplied.
    if pd.isna(
        features.get("avg_points_per_game", np.nan)
    ):
        points = features.get("points", np.nan)
        club_games = features.get(
            "club_games_played",
            np.nan
        )

        if (
            pd.notna(points)
            and pd.notna(club_games)
            and club_games > 0
        ):
            features["avg_points_per_game"] = (
                points / club_games
            )

    # --------------------------------------------------
    # Transfer history
    # --------------------------------------------------

    previous_transfers = features.get(
        "total_previous_transfers",
        np.nan
    )

    if pd.notna(previous_transfers):
        features["has_previous_transfer"] = int(
            previous_transfers > 0
        )

    # --------------------------------------------------
    # Historical market value
    # --------------------------------------------------

    previous_value = features.get(
        "previous_market_value_in_eur",
        np.nan
    )

    if (
        pd.notna(previous_value)
        and previous_value > 0
    ):
        features["log_previous_market_value"] = (
            np.log1p(previous_value)
        )
    else:
        features["log_previous_market_value"] = np.nan
        features["days_since_previous_valuation"] = np.nan

    # --------------------------------------------------
    # Guarantee all model columns exist
    # --------------------------------------------------

    all_model_features = list(
        dict.fromkeys(features_with_history)
    )

    model_features = {
        feature: features.get(feature, np.nan)
        for feature in all_model_features
    }

    return model_features